In [ ]:
import pandas as pd

In [ ]:
import os
os.chdir(r"C:\Users\HP\OneDrive\Documents\Data analytics\India-Digital-Spending-Tax-Health-Monitor")
from src.config import SNAPSHOT_DIR

In [ ]:
df=pd.read_csv(SNAPSHOT_DIR / "cpi_general_combined.csv")

In [ ]:
df

In [ ]:
df.shape


In [ ]:
df.size

In [ ]:
df.describe()

In [ ]:
set(df["month"])

In [ ]:
set(df["baseyear"])

In [ ]:
set(df["year"])

In [ ]:
set(df["month_code"])

In [ ]:
set(df["state"])

In [ ]:
set(df["sector"])

In [ ]:
set(df["group"])

In [ ]:
set(df["subgroup"])

In [ ]:
set(df["status"])

In [ ]:
df.info()

In [ ]:
by_state = df.groupby("state").agg(
    rows=("index", "size"),
    null_index=("index", lambda s: s.isna().sum()),
)
print(by_state.sort_values("null_index", ascending=False).head(8))

In [ ]:
# mask=df["state"]=="Arunachal Pradesh"
mask1 = (df["state"] == "Arunachal Pradesh") 
mask2 = df["index"].notna()

df[mask1 & mask2]

In [ ]:
from src.clean.state_utils import normalize_name, apply_mapping

df_copy = df.copy()
df_copy["state"] = df_copy["state"].apply(normalize_name)
df_copy = apply_mapping(df_copy, "cpi")
# g_copy.head()
set(df_copy["state"])


In [ ]:
dnh = df[df["state"] == "Dadra & Nagar Haveli"][["year", "month_code", "index"]]
dd = df[df["state"] == "Daman & Diu"][["year", "month_code", "index"]]

dnh = dnh.rename(columns={"index": "dnh_index"})
dd = dd.rename(columns={"index": "dd_index"})

both = dnh.merge(dd, on=["year", "month_code"])
both["diff"] = (both["dnh_index"] - both["dd_index"]).abs()

print(len(both))
print(both.head())
print(both["diff"].describe())

In [ ]:
cpi = df.copy()
cpi["year"] = cpi["year"]
cpi["quarter"] = (cpi["month_code"] - 1) // 3 + 1

quarterly = cpi.groupby(["state", "year", "quarter"], as_index=False).agg(
    cpi_index=("index", "mean"),
    n_months=("index", "count"),
)

quarterly.loc[quarterly["n_months"] < 2, "cpi_index"] = None

print(quarterly.head())
print(quarterly["n_months"].value_counts())

In [ ]:
prev = quarterly[["state", "year", "quarter", "cpi_index"]].copy()
prev["year"] = prev["year"] + 1
prev = prev.rename(columns={"cpi_index": "cpi_prev"})

q = quarterly.merge(prev, on=["state", "year", "quarter"], how="left")
q["cpi_yoy"] = (q["cpi_index"] / q["cpi_prev"] - 1) * 100

print(q["cpi_yoy"].describe())
print(q["cpi_yoy"].isna().sum())

In [ ]:
neg = q[q["cpi_yoy"] < 0][["state", "year", "quarter", "cpi_index", "cpi_prev", "cpi_yoy"]]
neg = neg.sort_values("cpi_yoy")
print(len(neg))
print(neg.head(10))

In [ ]:
two = q[q["state"].isin(["Dadra & Nagar Haveli", "Daman & Diu"]) & q["year"].isin([2018, 2019])]
two = two.sort_values(["year", "quarter", "state"])
print(two[["state", "year", "quarter", "cpi_yoy"]])

In [ ]:
import pandas as pd
from src.config import PROCESSED_DIR

df = pd.read_parquet(PROCESSED_DIR / "cpi_state_quarter.parquet")
print(df.duplicated(["state", "year", "quarter"]).sum())      # real data: should be 0

broken = pd.concat([df, df.head(1)])                           # add a copy of the first row
print(broken.duplicated(["state", "year", "quarter"]).sum())  # broken data: should be 1

In [ ]:
g = pd.read_parquet(PROCESSED_DIR / "gst_state_quarter.parquet")
print(g[g["total_filed"].isna()].groupby("state").size())